# S13 – Particle size harmonization and verification against the publications; origin and measurement of the dry diameter

In [1]:
import sys

import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from S13_comments import Audit, apply_comments   # the code of the comments of the step, apart from the data handling

INPUT = "../../S12_Coating_synthesis_and_row_identity/S12.xlsx"
OUTPUT = "../S13.xlsx"
COMMENTS = "../S13_comments.xlsx"
AUDIT = "../S13_audit.xlsx"
STEP = "S13"
GUL, LABOUTA, NANOPHAROS = "Gul. Et al..", "Labouta Et. Al. ", "Nanopharos"

df = pd.read_excel(INPUT, keep_default_na=False, na_values=[""]).astype(object)
print(len(df), "records before the step")
audit = Audit(STEP, df, before=INPUT)
put, drop = audit.put, audit.drop                  # a rule sets a value / removes records through the audit, which records the change
changes, removed = audit.changes, audit.removed    # the ledger of the step

3435 records before the step


### 1. Source sizes into the fields of the size block

In [2]:
for i in df.index:
    d, w, m = df.at[i, "Diameter (nm)"], df.at[i, "Size_in_Water (nm)"], df.at[i, "Size_in_Medium (nm)"]
    if df.at[i, "_src"] == GUL and not pd.isna(d):
        put(i, 'Dry_Diameter (nm)', d, f"{STEP}_tem_from_diameter",
            "Gul 'Diameter' is a primary (microscopy) size unless the publication review says otherwise", outside=False)
    if not pd.isna(w):
        put(i, "DLS_Diameter_Water (nm)", w, f"{STEP}_dls_water", "source size in water -> DLS water", outside=False)
    if not pd.isna(m):
        put(i, "DLS_Diameter_Medium (nm)", m, f"{STEP}_dls_medium", "source size in medium -> DLS medium", outside=False)
    sel = m if not pd.isna(m) else w
    if not pd.isna(sel):
        put(i, "DLS_Diameter (nm)", sel, f"{STEP}_dls_selected", "selected DLS diameter = medium value (exposure-relevant), else water", outside=False)

### 2. Values from the publication review: the comments of this step

In [3]:
# every value from the publication review: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S13_curated_L1_doi', 'S13_curated_L2_material', 'S13_curated_L3_coat', 'S13_curated_L4_diameter', 'S13_curated_L5_cell', 'S13_curated_L7_row'])

5400 comments applied (S13_curated_L1_doi, S13_curated_L2_material, S13_curated_L3_coat, S13_curated_L4_diameter, S13_curated_L5_cell, S13_curated_L7_row): 5400 values set, 0 records removed


### 3. Corrections from the check against the publications: the comments of this step

In [4]:
# every correction from the check of this field against the publications: the rows of these rules in the comments table (record, variable, new value or removal, comment)
apply_comments(audit, COMMENTS, ['S13_publication_check_01', 'S13_publication_check_02', 'S13_publication_check_03', 'S13_publication_check_04', 'S13_publication_check_05', 'S13_publication_check_06', 'S13_publication_check_07', 'S13_publication_check_08', 'S13_publication_check_09', 'S13_publication_check_10', 'S13_publication_check_11', 'S13_publication_check_12', 'S13_publication_check_13', 'S13_publication_check_14', 'S13_publication_check_15', 'S13_publication_check_16', 'S13_publication_check_17', 'S13_publication_check_18', 'S13_publication_check_19', 'S13_publication_check_20', 'S13_publication_check_21', 'S13_publication_check_22', 'S13_publication_check_23', 'S13_publication_check_24', 'S13_publication_check_25', 'S13_publication_check_26', 'S13_publication_check_27', 'S13_publication_check_28', 'S13_publication_check_29', 'S13_publication_check_30', 'S13_publication_check_31', 'S13_publication_check_32', 'S13_publication_check_33', 'S13_publication_check_34', 'S13_publication_check_35', 'S13_publication_check_36', 'S13_publication_check_37', 'S13_publication_check_38', 'S13_publication_check_39', 'S13_publication_check_40', 'S13_publication_check_41'])

1890 comments applied (S13_publication_check_01, S13_publication_check_02, S13_publication_check_03, S13_publication_check_04, S13_publication_check_05, S13_publication_check_06, S13_publication_check_07, S13_publication_check_08, S13_publication_check_09, S13_publication_check_10, S13_publication_check_11, S13_publication_check_12, S13_publication_check_13, S13_publication_check_14, S13_publication_check_15, S13_publication_check_16, S13_publication_check_17, S13_publication_check_18, S13_publication_check_19, S13_publication_check_20, S13_publication_check_21, S13_publication_check_22, S13_publication_check_23, S13_publication_check_24, S13_publication_check_25, S13_publication_check_26, S13_publication_check_27, S13_publication_check_28, S13_publication_check_29, S13_publication_check_30, S13_publication_check_31, S13_publication_check_32, S13_publication_check_33, S13_publication_check_34, S13_publication_check_35, S13_publication_check_36, S13_publication_check_37, S13_publication

### 4. Origin of the dry diameter (stated in the publication, unless listed here)

In [5]:
has = ~df['Dry_Diameter (nm)'].map(pd.isna)
doi = df["DOI"].astype(str).str.strip().str.lower()
DRY_ORIGIN = [   # publication, material ("" = every material), origin of the dry diameter
    ('10.1038/s41598-018-19628-z', '', 'manufacturer declaration'),
    ('10.1016/j.tiv.2016.09.012', 'Au', 'manufacturer'),
    ('10.3390/ijms140815546', 'Fe3O4', 'manufacturer'),
    ('10.1016/j.tiv.2014.06.005', 'Ag', 'manufacturer'),
    ('10.1166/jnn.2011.4250', 'ZnO', "authors' earlier study"),
    ('10.1177/0960327112459206', 'SiO2', "authors' earlier study"),
    ('10.1080/17458080.2015.1014934', 'Au', "authors' earlier study"),
    ('10.1016/j.dental.2015.02.009', '', 'manufacturer'),
    ('10.2147/IJN.S208225', 'Pt', 'manufacturer'),
    ('10.2147/IJN.S208225', 'SiO2', 'manufacturer'),
    ('10.1002/ppsc.201300357', 'Au', "authors' earlier study"),
    ('10.3109/17435390.2016.1141338', 'TiO2', "authors' earlier study"),
    ('10.3762/bjnano.6.28', 'Au', "authors' earlier study"),
    ('10.2147/IJN.S214008', '', 'manufacturer declaration'),
    ('10.1007/s00204-012-0840-4', '', "authors' earlier study"),
    ('10.1021/tx400231u', 'TiO2', 'manufacturer declaration'),
]
df["Dry_Diameter_Source"] = None
df.loc[has, "Dry_Diameter_Source"] = "stated in the publication"
for d, material, source in DRY_ORIGIN:
    m = has & (doi == d.lower())
    if material:
        m &= df["Material"].astype(str) == material
    df.loc[m, "Dry_Diameter_Source"] = source

### 5. Measuring technique of the dry diameter of each publication (and material, where one publication used several)

In [6]:
DRY_TECHNIQUE = {   # technique -> publications
    'TEM': [
        '10.1002/aoc.5308', '10.1002/jat.2953', '10.1002/jat.3213',
        '10.1002/jbm.a.35491', '10.1002/jbm.b.33068', '10.1002/ppsc.201300357',
        '10.1002/smll.201502346', '10.1007/s10565-011-9191-9', '10.1007/s10856-012-4716-1',
        '10.1007/s10876-019-01695-5', '10.1007/s11051-014-2726-2', '10.1007/s12010-012-9699-3',
        '10.1016/j.bbrc.2010.04.156', '10.1016/j.ceramint.2013.02.026', '10.1016/j.chemosphere.2016.11.018',
        '10.1016/j.colsurfb.2017.12.042', '10.1016/j.dental.2015.02.009', '10.1016/j.ecoenv.2019.109770',
        '10.1016/j.fct.2012.01.017', '10.1016/j.jhazmat.2009.12.089', '10.1016/j.jphotobiol.2017.04.004',
        '10.1016/j.matdes.2019.108187', '10.1016/j.mrfmmm.2014.08.001', '10.1016/j.procbio.2019.08.004',
        '10.1016/j.scitotenv.2011.07.039', '10.1016/j.taap.2011.12.020', '10.1016/j.talanta.2013.01.037',
        '10.1016/j.tiv.2009.06.001', '10.1016/j.tiv.2011.12.004', '10.1016/j.tiv.2012.08.021',
        '10.1016/j.tiv.2014.06.005', '10.1016/j.tiv.2014.12.007', '10.1016/j.tiv.2016.09.012',
        '10.1016/j.toxlet.2010.10.003', '10.1016/j.toxlet.2012.09.025', '10.1016/j.toxrep.2014.11.002',
        '10.1021/nn3010087', '10.1021/nn304439f', '10.1021/tx3002093',
        '10.1021/tx400231u', '10.1038/s41598-018-19628-z', '10.1039/c0nr00204f',
        '10.1080/01480545.2016.1199563', '10.1080/01480545.2018.1452930', '10.1080/17458080.2015.1014934',
        '10.1093/mutage/gex027', '10.1109/TNB.2003.820277', '10.1155/2012/734398',
        '10.1155/2016/4023852', '10.1155/2018/1367801', '10.1166/jnn.2011.4250',
        '10.1177/0748233713511512', '10.1177/0960327112459206', '10.1177/1091581813487563',
        '10.1177/1559325818807382', '10.1186/s12951-019-0456-4', '10.1186/s12989-016-0160-6',
        '10.1371/journal.pone.0069534', '10.1371/journal.pone.0099175', '10.2147/IJN.S13244',
        '10.2147/IJN.S72580', '10.3109/15376516.2015.1070229', '10.3109/17435390.2010.528847',
        '10.3109/17435390.2013.815284', '10.3109/17435390.2016.1141338', '10.3390/cells8050444',
        '10.3390/ijms140815546', '10.5402/2012/454072', '10.1594/ecr2013/C-2054',
        '10.2147/IJN.S214008', '10.1016/j.tiv.2011.02.015', '10.1007/s00449-009-0354-5',
        '10.1007/s11051-013-2229-6', '10.1007/s00204-012-0840-4',
    ],
    'STEM': [
        '10.1002/jat.3566', '10.1007/s11051-018-4406-0', '10.1016/j.toxlet.2016.03.005',
        '10.1186/1743-8977-6-18',
    ],
    'SEM': [
        '10.1002/jbm.a.35518', '10.1039/B914875B', '10.1155/2012/160145',
        '10.1016/j.tiv.2011.07.010',
    ],
    'TEM/SEM': [
        '10.1007/s10856-019-6221-2', '10.1007/s11051-018-4333-0', '10.1016/j.nano.2014.01.006',
        '10.1016/j.nano.2015.03.004', '10.3109/17435390.2011.625127', '10.4172/2157-7439.1000194',
    ],
    'FE-SEM': [
        '10.1016/j.ceramint.2014.09.095', '10.1016/j.jphotobiol.2020.111778', '10.1155/2019/5287632',
    ],
    'HR-TEM': [
        '10.1186/s11671-019-2967-1',
    ],
}
DRY_TECHNIQUE_BY_MATERIAL = [   # publication, material, technique (overrides the publication-wide technique)
    ('10.1016/j.btre.2014.08.002', 'Ag', 'HR-TEM'),
    ('10.1016/j.btre.2014.08.002', 'Au', 'TEM'),
    ('10.2147/IJN.S208225', 'Pt', 'TEM'),
    ('10.2147/IJN.S208225', 'SiO2', 'TEM'),
    ('10.2147/IJN.S208225', 'TiO2', 'TEM'),
    ('10.3762/bjnano.6.28', 'Au', 'TEM'),
    ('10.3762/bjnano.6.28', 'Fe3O4', 'TEM'),
    ('10.3762/bjnano.6.28', 'MnO', 'TEM'),
    ('10.1021/tx400231u', 'TiO2', None),   # no technique: the manufacturer's size of Aeroxide P25 is given without a measuring method
]
df["Dry_Diameter_Method"] = None
for method, dois in DRY_TECHNIQUE.items():
    df.loc[has & doi.isin([d.lower() for d in dois]), "Dry_Diameter_Method"] = method
for d, material, method in DRY_TECHNIQUE_BY_MATERIAL:
    df.loc[has & (doi == d.lower()) & (df["Material"].astype(str) == material), "Dry_Diameter_Method"] = method

### The table after the step and its audit

In [7]:
# the table after the step; the audit module compares it with the table before the step and writes the audit workbook
df.to_excel(OUTPUT, index=False)
audit.write(AUDIT)

1 records removed and 13307 cells changed compared with the table before the step


3434 records after the step | 1 records removed | 14488 cells changed | ../S13_audit.xlsx
